In [69]:
import sys
import os

sibling_dir = os.path.abspath('../02_vector_search/hw_vector_search')

if sibling_dir not in sys.path:
    sys.path.append(sibling_dir)

In [70]:
from pathlib import Path

# Add the parent directory (04_eval) to Python's import path
sys.path.append(str(Path.cwd().parent))

In [71]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# Generating questions
# Q1 What's the average number of input tokens across these 3 calls?

In [72]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [73]:
documents1 = next(
    doc for doc in documents
    if doc["filename"] == "01-agentic-rag/lessons/01-intro.md"
)

print(documents1["filename"])

01-agentic-rag/lessons/01-intro.md


In [74]:
documents2 = next(
    doc for doc in documents
    if doc["filename"] == "01-agentic-rag/lessons/02-environment.md"
)

print(documents2["filename"])

01-agentic-rag/lessons/02-environment.md


In [75]:
documents3 = next(
    doc for doc in documents
    if doc["filename"] == "01-agentic-rag/lessons/03-rag.md"
)

print(documents3["filename"])

01-agentic-rag/lessons/03-rag.md


In [76]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [77]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [78]:
from evaluation_utils import llm_structured
import json

In [79]:
user_prompt_1 = json.dumps({"filename": documents1['filename'], "content": documents1['content']})
user_prompt_2 = json.dumps({"filename": documents2['filename'], "content": documents2['content']})
user_prompt_3 = json.dumps({"filename": documents3['filename'], "content": documents3['content']})

In [80]:
result_1,usage_1=llm_structured(openai_client, data_gen_instructions, user_prompt_1, Questions)
result_2,usage_2=llm_structured(openai_client, data_gen_instructions, user_prompt_2, Questions)
result_3,usage_3=llm_structured(openai_client, data_gen_instructions, user_prompt_3, Questions)

In [81]:
token_1 = usage_1.input_tokens
token_2 = usage_2.input_tokens
token_3 = usage_3.input_tokens

In [82]:
# calulate the average number of input tokens used for generating questions

average_tokens = (token_1 + token_2 + token_3) / 3
print(f"Average input tokens used for generating questions: {average_tokens}")

Average input tokens used for generating questions: 1354.0


# Q2 First result with text search

In [83]:
import pandas as pd
ground_truth = pd.read_csv("hw_data/hw_ground_truth.csv").to_dict(orient="records")
print(len(ground_truth))
print(ground_truth[0])

360
{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?", 'filename': '01-agentic-rag/lessons/01-intro.md'}


In [84]:
# create chunks of documents for evaluation

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [85]:
len(chunks)

295

In [86]:
from minsearch import Index, VectorSearch

text_index = Index(text_fields=['content'], keyword_fields=['filename'])
text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [87]:
parent_dir = os.path.abspath("..")

if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [88]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [89]:
from hw_embedder import Embedder
embd = Embedder()

In [90]:
def vector_search(query, num_results=5):
    v = embd.encode(query)
    return vector_index.search(v, num_results=num_results)

vector_index = VectorSearch(keyword_fields=['filename'])
vector_index.fit(embd.encode_batch([c['content'] for c in chunks]), chunks)

In [91]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [92]:
text_search(q)[0]

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

In [93]:
print("text_search:", text_search(q)[0]['filename'])

text_search: 01-agentic-rag/lessons/03-rag.md


# Q3 First result with vector search

In [94]:
vector_search(q)[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [95]:
print("vector_search:", vector_search(q)[0]['filename'])

vector_search: 01-agentic-rag/lessons/01-intro.md


# Q4 Evaluating text search

In [96]:
# define compute_relevance function to calculate the relevance of search results based on ground truth

def compute_relevance(gt, search_function):
    relevance = []
    for item in gt:
        results = search_function(item['question'])
        relevance.append([res['filename'] == item['filename'] for res in results])
    return relevance

In [97]:
# define hit rate function to calculate the proportion of queries that returned at least one relevant result

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [98]:
# define mrr function to calculate the mean reciprocal rank of search results based on ground truth

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [99]:
# performing evaluation of the search function using the ground truth data

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [100]:
compute_relevance_textsearch = compute_relevance(ground_truth, text_search)

In [101]:
res_hit=hit_rate(compute_relevance_textsearch)

In [102]:
print(f"Hit rate for text search: {res_hit}")

Hit rate for text search: 0.7583333333333333


# Q5. Evaluating vector search 

In [103]:
compute_relevance_vsearch = compute_relevance(ground_truth, vector_search)

In [104]:
mrr_vsearch = mrr(compute_relevance_vsearch)

In [105]:
print(f"mrr for vector search: {mrr_vsearch}")

mrr for vector search: 0.5486111111111112


# Q6. Tuning hybrid search

In [106]:
# define rrf function 

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [107]:
# define hybrid search function that combines text search and vector search results using reciprocal rank fusion

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [108]:
k_values = [1, 50, 100, 200]

In [109]:
for k in k_values:
    fun = lambda q, k=k: hybrid_search(q, k=k)
    metrics = evaluate(ground_truth, fun)
    print(k, metrics)

1 {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}
50 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
100 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
200 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}
